# Module 7 — Unity Catalog Basics (Governance)
Exam domain: **Security, Governance, Monitoring & Testing**

Requires a Databricks workspace with **Unity Catalog enabled**. If your
workspace still uses the legacy `hive_metastore`, read this as reference —
some commands below will fail without a UC-enabled metastore attached.

In [ ]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "module7")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE {catalog}.{schema}")

In [ ]:
df = spark.createDataFrame([(1, 100.0), (2, 50.0)], ["order_id", "amount"])
df.write.format("delta").mode("overwrite").saveAsTable("orders")
spark.sql("SELECT * FROM orders").show()

## Grants (adjust group names to ones that exist in your workspace)

In [ ]:
%sql
-- GRANT USE CATALOG ON CATALOG main TO `analysts`;
-- GRANT USE SCHEMA ON SCHEMA main.module7 TO `analysts`;
-- GRANT SELECT ON TABLE main.module7.orders TO `analysts`;
SHOW GRANTS ON TABLE orders;

## Inspecting lineage and table info

In [ ]:
%sql
DESCRIBE EXTENDED orders;

## Managed vs. external table
Managed tables store data under the schema's managed storage location; dropping
the table deletes the data. External tables are created with `LOCATION` and
keep their files on `DROP TABLE`.

In [ ]:
%sql
-- CREATE TABLE orders_external
-- LOCATION 's3://my-bucket/module7/orders_external'
-- AS SELECT * FROM orders;

## Row filters / column masks (dynamic views pattern)

In [ ]:
%sql
-- ALTER TABLE orders SET ROW FILTER region_filter ON (region);
-- ALTER TABLE orders ALTER COLUMN amount SET MASK mask_amount;